In [1]:
import sys
print("The python version ishehe: " + sys.version)

The python version ishehe: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


# AdvTG — end-to-end pipeline

Adversarial HTTP traffic generation vs. DL malicious-traffic detectors, run stage by stage:

1. **Dataset** → 2. **Detectors** (token + image) → 3. **LLM finetune** (QLoRA) → 4. **PPO** adversarial generation

**Where this runs:** a **Kaggle notebook** (GPU). The cells run on Kaggle's VM, not your laptop.

- Notebook ▸ Settings: **Accelerator = GPU T4 x2** (avoid P100: current torch/bitsandbytes builds don't support it), **Internet = On** (needed for git clone, pip, and HF model downloads).
- Runs on Kaggle's default Python. The stack is modern and unpinned, so you don't need a special runtime, a torch downgrade, or kernel restarts.
- The VM only has what's in the **git remote**. **Push your branch first**; the Setup cell clones it into `/kaggle/working/AdvTG` (and `git pull`s on re-runs). Local edits do **not** sync to the VM.
- Storage: the repo, `dataset/` and `model/` live under **`/kaggle/working`**, which is kept as notebook output when you *Save Version* (20 GB limit). HF model weights are cached outside it, so they don't count against that limit.
- Optional: add an **`HF_TOKEN`** under Add-ons ▸ Secrets. It gives faster, rate-limit-free HF downloads.
- Only GPU 0 is used (`CUDA_VISIBLE_DEVICES=0`). Everything fits on one T4, and this avoids multi-GPU surprises.

**Each stage installs its own deps** at the top of its first cell (there's no central install step), so you can run any stage on its own. `requirements.txt` still lists the full stack for venv/Docker use. Run **Setup + Config** once, then run the stages in order.

In [ ]:
import os, sys, subprocess

# Kaggle T4 x2: pin to one GPU (must be set before torch initialises CUDA).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# HF token from Kaggle Secrets (Add-ons ▸ Secrets ▸ HF_TOKEN) — optional.
if "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass   # no secret attached -> unauthenticated HF downloads

# Setup as a function. Light + idempotent (git pull + chdir + sys.path — NO torch import),
# so it's safe to call at the top of any stage via prepare(). Run this cell once to define it.
GIT_URL = "https://github.com/TejaswiMN/AdvTG.git"
BRANCH  = "modernize"
WORK    = "/kaggle/working"            # persisted as notebook output on Save Version

def setup():
    """Clone/pull the repo, cd into it, make it importable. Sets global REPO."""
    global REPO
    if os.path.isfile(os.path.join(os.getcwd(), "gen_synthetic_data.py")):
        REPO = os.getcwd()                                   # already inside the repo
    else:
        # clone target on the hosted VM (Kaggle)
        REPO = os.path.join(WORK, "AdvTG")
        if os.path.isdir(os.path.join(REPO, ".git")):
            subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=False)
            subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=False)
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
        else:
            subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, REPO], check=True)
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)                             # make the DL package importable
    return REPO

setup()
print("repo:", REPO, "| has gen_synthetic_data.py:", os.path.isfile("gen_synthetic_data.py"))

## Config — the settings live in `config()`

To change a setting (dataset source, sizes, epochs…), edit the value **inside the `config()` function** below and re-run this cell. Everything downstream reads `dataset/train_data2.json`.

`setup()` and `config()` are light, torch-free helpers; `prepare()` runs both and is called at the top of every stage — so each stage is self-contained and you never have to scroll back up to re-run Setup/Config.

In [ ]:
import os

# Config as a function (light + idempotent, no torch import). prepare() = setup() + config(),
# called at the top of every stage so each stage is self-contained. To change a setting,
# edit the value inside config() and re-run this cell.
def config():
    """Define dataset/training settings + paths as globals (needs REPO from setup())."""
    global DATASET_SOURCE, N_TRAIN, N_TEST, MALICIOUS_RATIO
    global KAGGLE_DATASET, CICIDS_INPUT, CICIDS_MAX_PER_CLASS
    global MAX_LENGTH, BATCH_SIZE, NUM_EPOCHS, TRAIN_BERT
    global DATA_DIR, MODEL_DIR, TRAIN_JSON, TEST_JSON

    # ─── the one knob: where dataset/train_data2.json comes from ──────────────
    DATASET_SOURCE  = "synthetic"      # "synthetic" | "cicids2017"
    # synthetic sizes
    N_TRAIN         = 20000
    N_TEST          = 4000
    MALICIOUS_RATIO = 0.35
    # CIC-IDS2017 (only used when DATASET_SOURCE == "cicids2017").
    # Easiest on Kaggle: Add Input ▸ this dataset -> it mounts read-only at CICIDS_INPUT.
    KAGGLE_DATASET       = "chethuhn/network-intrusion-dataset"
    CICIDS_INPUT         = "/kaggle/input/network-intrusion-dataset"
    CICIDS_MAX_PER_CLASS = 20000
    # detector training
    MAX_LENGTH = 512
    BATCH_SIZE = 16
    NUM_EPOCHS = 2
    TRAIN_BERT = False                 # heavy, and not used by the PPO reward
    # paths (match the repo's ../dataset and ../model conventions) — under /kaggle/working/AdvTG
    DATA_DIR   = os.path.join(REPO, "dataset")
    MODEL_DIR  = os.path.join(REPO, "model")
    TRAIN_JSON = os.path.join(DATA_DIR, "train_data2.json")
    TEST_JSON  = os.path.join(DATA_DIR, "test2.json")
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

def prepare():
    """setup() + config() — call once at the top of any stage to make it self-contained."""
    setup(); config()

config()
print("dataset source:", DATASET_SOURCE, "| model dir:", MODEL_DIR)

## Stage 1 — Dataset

Produces `dataset/train_data2.json` (+ `test2.json`) — the single file every downstream stage reads.

In [ ]:
import shutil
prepare()   # re-establish repo + config, so this stage runs standalone

if DATASET_SOURCE == "synthetic":
    !python gen_synthetic_data.py --n {N_TRAIN} --test-n {N_TEST} \
        --malicious-ratio {MALICIOUS_RATIO} --out "{TRAIN_JSON}" --test-out "{TEST_JSON}"

elif DATASET_SOURCE == "cicids2017":
    !apt-get -qq install -y tshark >/dev/null
    if os.path.isdir(CICIDS_INPUT):
        CICIDS_DIR = CICIDS_INPUT                      # attached via Add Input (no download)
    else:
        # fallback: download via the API (needs Kaggle Secrets KAGGLE_USERNAME / KAGGLE_KEY).
        # /tmp, not /kaggle/working — the raw PCAPs would eat the 20 GB output quota.
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"]      = _s.get_secret("KAGGLE_KEY")
        CICIDS_DIR = "/tmp/cicids"
        !pip -q install kaggle
        !kaggle datasets download -d {KAGGLE_DATASET} -p {CICIDS_DIR} --unzip
    # PCAPs carry the HTTP text; TrafficLabelling CSVs carry the labels (join on the 5-tuple).
    !python build_train_data.py \
        --pcap-dir "{CICIDS_DIR}/PCAPs" \
        --csv-dir  "{CICIDS_DIR}/TrafficLabelling" \
        --max-per-class {CICIDS_MAX_PER_CLASS} \
        --out "{TRAIN_JSON}"
    shutil.copy(TRAIN_JSON, TEST_JSON)   # PPO reads a held-out file; reuse the extract

else:
    raise ValueError("DATASET_SOURCE must be 'synthetic' or 'cicids2017'")

In [5]:
import json, collections
recs = json.load(open(TRAIN_JSON, encoding="utf-8"))
print(len(recs), "records", dict(collections.Counter(r["Label"] for r in recs)))
r = recs[0]
print("\n" + r["Request Line"])
for k, v in list(r["Request Headers"].items())[:4]:
    print(f"  {k}: {v}")
print("  ->", r["Label"], "| source:", r["Source"])

20000 records {'Benign': 13000, 'Malicious': 7000}

GET /blog/2024/05/release-notes HTTP/1.1
  Host: shop.example.com
  User-Agent: Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0
  Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8
  Accept-Language: es-ES,es;q=0.9
  -> Benign | source: synthetic-browse


## Stage 2 — Detector training

Trains the token-level (TextCNN / CNN-LSTM / DNN) and image detectors, then writes the `model_configs` pickles the PPO stage attacks. Runs on CPU or GPU.

In [6]:
# token-level detectors (TextCNN / CNN-LSTM / DNN), sharing the BERT tokenizer's vocab
prepare()   # re-establish repo + config (must run before the DL imports below)
# Stage 2 deps (torch/numpy come from the runtime):
!pip -q install "transformers>=4.46" "datasets>=2.20" "scikit-learn>=1.3"
import os, torch
import DL.training as _T
from transformers import TrainingArguments, AutoTokenizer
from DL.data_processing import load_data, prepare_dataset
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.training import train_custom_model, train_transformer_model

_T.MODEL_PATH = MODEL_DIR          # repo hardcodes ./models/; redirect saves under model/
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

TOKENIZER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
data = load_data(TRAIN_JSON)
train_ds, val_ds, test_ds = prepare_dataset(data, tokenizer, MAX_LENGTH)

vocab_size, embed_size, num_classes = len(tokenizer.vocab), 128, 2
os.makedirs(os.path.join(MODEL_DIR, "custom_models"), exist_ok=True)
args = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "custom_models"),
                         per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                         learning_rate=2e-5, num_train_epochs=NUM_EPOCHS, report_to="none")

for name, model in {
        "textcnn":  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "cnn_lstm": CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "dnn":      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)}.items():
    print("training", name)
    train_custom_model(model, name, train_ds, val_ds, args)

if TRAIN_BERT:
    bargs = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "bert"),
                              evaluation_strategy="epoch", learning_rate=2e-5,
                              per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                              num_train_epochs=NUM_EPOCHS, weight_decay=0.01, save_strategy="epoch",
                              load_best_model_at_end=True, report_to="none")
    train_transformer_model("bert", TOKENIZER_NAME, train_ds, val_ds, bargs)

device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

training textcnn
Epoch 1, Eval Loss: 23.006947368383408, Accuracy: 0.93375, Precision: 0.9385023164647184, Recall: 0.93375, F1: 0.931784134917312
Epoch 2, Eval Loss: 11.835430413484573, Accuracy: 0.98875, Precision: 0.9888861739828819, Recall: 0.98875, F1: 0.9887045106275818
training cnn_lstm
Epoch 1, Eval Loss: 3.913216294720769, Accuracy: 0.998125, Precision: 0.9981302179962894, Recall: 0.998125, F1: 0.998123620070349
Epoch 2, Eval Loss: 0.562031286302954, Accuracy: 1.0, Precision: 1.0, Recall: 1.0, F1: 1.0
training dnn
Epoch 1, Eval Loss: 10.230498161166906, Accuracy: 0.970625, Precision: 0.9709199867052359, Recall: 0.970625, F1: 0.9704032970537483
Epoch 2, Eval Loss: 4.180680561577901, Accuracy: 0.991875, Precision: 0.9918946637550882, Recall: 0.991875, F1: 0.9918808975486898


In [7]:
# image-based detectors: each request rendered as a 28x28 byte image (ord(c) % 128)
import numpy as np, torch, os
from torch.utils.data import DataLoader, TensorDataset
from DL.data_processing import json_to_string
from DL.image_models import ImageCNN, ImageMLP

IMG = (28, 28)
def to_image(it):
    text = it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
    v = [ord(c) % 128 for c in text][:IMG[0] * IMG[1]]
    v += [0] * (IMG[0] * IMG[1] - len(v))
    return np.array(v, dtype=np.float32).reshape(IMG)

X = torch.tensor(np.stack([to_image(r) for r in data]))
y = torch.tensor([1 if r["Label"] == "Malicious" else 0 for r in data], dtype=torch.long)
k = int(len(X) * 0.9)
loader = DataLoader(TensorDataset(X[:k], y[:k]), batch_size=64, shuffle=True)

for name, m in {"imagecnn": ImageCNN(), "imagemlp": ImageMLP()}.items():
    m.to(device); opt = torch.optim.Adam(m.parameters(), 1e-3); lf = torch.nn.CrossEntropyLoss()
    for _ in range(int(NUM_EPOCHS)):
        m.train()
        for xb, yb in loader:
            loss = lf(m(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        acc = (m(X[k:].to(device)).argmax(1).cpu() == y[k:]).float().mean().item()
    path = os.path.join(MODEL_DIR, "custom_models", name + ".bin")
    torch.save(m.state_dict(), path)
    print(f"{name}: val acc {acc:.3f} -> {path}")

imagecnn: val acc 0.958 -> /content/AdvTG/model/custom_models/imagecnn.bin
imagemlp: val acc 0.891 -> /content/AdvTG/model/custom_models/imagemlp.bin


In [8]:
import pickle, os
from transformers import AutoTokenizer
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.image_models import ImageCNN, ImageMLP

cm = os.path.join(MODEL_DIR, "custom_models")

def cfg(name, cls):
    return {"type": "custom", "name": name, "path": os.path.join(cm, name + ".bin"), "class": cls}

text_configs = [
    cfg("textcnn",  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("cnn_lstm", CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("dnn",      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
]
image_configs = [cfg("imagecnn", ImageCNN()), cfg("imagemlp", ImageMLP())]

pickle.dump(text_configs,  open(os.path.join(MODEL_DIR, "model_configs.pkl"), "wb"))
pickle.dump(image_configs, open(os.path.join(MODEL_DIR, "imgae_model_configs.pkl"), "wb"))

# PPO's Text reward re-tokenises responses with this — save it even when BERT is skipped.
AutoTokenizer.from_pretrained(TOKENIZER_NAME).save_pretrained(os.path.join(MODEL_DIR, "bert"))
print("wrote model_configs.pkl, imgae_model_configs.pkl, model/bert/ tokenizer")

wrote model_configs.pkl, imgae_model_configs.pkl, model/bert/ tokenizer


## Stage 3 — LLM finetuning (QLoRA, GPU)

4-bit LoRA SFT of Llama-3-8b — plain **peft + bitsandbytes** (no unsloth) — to generate benign/malicious
traffic in the dataset's format. Fits a T4 (MAX_SEQ 1024, batch 1 + grad-accum). Uses TRL **`SFTConfig`**;
saves the adapter to `model/llama_lora`.

In [9]:
import os, json, torch
prepare()   # re-establish repo + config, so this stage runs standalone
# Stage 3 deps — plain peft + bitsandbytes QLoRA (NO unsloth); trl>=0.11 gives SFTConfig.
# (transformers + datasets were already installed by Stage 2.)
!pip -q install "trl>=0.11" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.34"
assert torch.cuda.is_available(), "Stage 3 needs a GPU runtime"

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

MODEL   = "unsloth/llama-3-8b-bnb-4bit"   # ungated, pre-4bit checkpoint; loads via transformers+bitsandbytes (no unsloth lib)
MAX_SEQ = 1024                            # plain peft is heavier than unsloth -> trim from 2048 for the T4
_bf16   = torch.cuda.is_bf16_supported()  # T4 -> False (fp16); L4/A100 -> True (bf16)

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", dtype=torch.float16)
model = prepare_model_for_kbit_training(model)     # freeze base, cast norms to fp32, enable input grads
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def json_to_string(d, indent=0):
    out, pad = [], " " * indent
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, (dict, list)):
                out.append(f"{pad}{k}:"); out.append(json_to_string(v, indent + 1))
            else:
                out.append(f"{pad}{k}: {v}")
    elif isinstance(d, list):
        for it in d:
            out.append(json_to_string(it, indent))
    else:
        out.append(f"{pad}{d}")
    return "\n".join(out)

alpaca = ("Below is an instruction that describes a task, paired with an input that provides "
          "further context. Write a response that appropriately completes the request.\n\n"
          "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n{}")
EOS = tok.eos_token
data = json.load(open(TRAIN_JSON, encoding="utf-8"))
def body(it): return it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
texts = [alpaca.format(
            "Follow these tips to generate malicious http traffic" if it["Label"] == "Malicious"
            else "Follow these tips to generate benign http traffic",
            it["Request Line"], body(it)) + EOS
         for it in data]
ds = Dataset.from_dict({"text": texts}).shuffle(seed=42)

# Current TRL: dataset_text_field / max_seq_length / packing live on SFTConfig (subclasses TrainingArguments).
trainer = SFTTrainer(
    model=model, processing_class=tok, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ, packing=False,
        per_device_train_batch_size=1, gradient_accumulation_steps=16,   # bs=1 fits the T4; effective batch 16
        warmup_steps=5, max_steps=60, learning_rate=2e-4,
        fp16=not _bf16, bf16=_bf16,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5, optim="paged_adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407,
        output_dir=os.path.join(MODEL_DIR, "llama_outputs"), report_to="none"))
trainer.train()
model.save_pretrained(os.path.join(MODEL_DIR, "llama_lora"))
tok.save_pretrained(os.path.join(MODEL_DIR, "llama_lora"))
print("saved LoRA ->", os.path.join(MODEL_DIR, "llama_lora"))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


Adding EOS to train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Step,Training Loss
5,2.145013
10,1.435752
15,1.109456
20,0.980527
25,0.855772
30,0.875138
35,0.909535
40,0.839209
45,0.900414
50,0.912280


saved LoRA -> /content/AdvTG/model/llama_lora


## Stage 4 — PPO adversarial generation (GPU)

Hand-rolled PPO in `RL-Adv/ppo_core.py` (**no trl-PPO** — immune to trl API churn) tunes a generator
to flip the frozen detectors' predictions. Reward = mean detector probability of the *opposite*
label. `policy="pythia"` is the cheap demo generator; **Stage 4b** runs the Stage-3 Llama (paper-faithful).

In [ ]:
import os, sys, importlib
prepare()   # re-establish repo + config, so this stage runs standalone
# deps already installed by Stage 2 (transformers, datasets); torch comes from the runtime.
RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)                       # so ppo_core's ../model and ../dataset paths resolve
import ppo_core; importlib.reload(ppo_core)

# policy="pythia" -> cheap demo generator (EleutherAI/pythia-160m). FEATURE_TYPE: "Text" | "Image".
# Tune the knobs (steps / sample_size / batch_size) as needed; batch_size must be 1..4.
asr = ppo_core.train_ppo(policy="pythia", feature_type="Text",
                         steps=40, sample_size=2000, batch_size=4)
os.chdir(REPO)
print("Stage 4 (pythia) ASR:", asr)

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  375MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

  [ckpt] step 30 -> ../model/ppo_model/Text
step 030  reward 0.439  target-hit 0.75  loss 0.382
step 031  reward 0.065  target-hit 0.50  loss 0.258
step 032  reward -1.167  target-hit 0.00  loss -0.768
step 033  reward 0.574  target-hit 0.75  loss -2.080
step 034  reward -0.388  target-hit 0.50  loss -2.267
step 035  reward 0.333  target-hit 0.75  loss -0.030
step 036  reward -0.418  target-hit 0.25  loss 0.147
step 037  reward 0.114  target-hit 0.50  loss 0.035
step 038  reward 0.772  target-hit 1.00  loss 0.158
step 039  reward 0.346  target-hit 0.75  loss -0.069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [ckpt] step 40 -> ../model/ppo_model/Text


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[done] trained to step 40; checkpoint -> ../model/ppo_model/Text


In [ ]:
import os
for p in [os.path.join(REPO, "dataset", "train_data2.json"),
          os.path.join(REPO, "model", "model_configs.pkl"),
          os.path.join(REPO, "model", "llama_lora", "adapter_config.json")]:
    print(os.path.exists(p), p)

In [ ]:
import glob, json, os

files = sorted(glob.glob(os.path.join(REPO, "dataset", "PPO_data", "**", "*.json"), recursive=True))
if files:
    print("latest:", files[-1])
    print(json.dumps(json.load(open(files[-1]))[:2], indent=2)[:1500])
else:
    print("no PPO_data yet — run Stage 4 first")

## Stage 4b (faithful): PPO-tune the Stage-3 Llama instead of pythia-160m

In [ ]:
import torch
print(f"GPU used: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import os
lora = os.path.join(REPO, "model", "llama_lora")
ckpt = os.path.join(REPO, "model", "ppo_llama_ckpt")
print("Stage 3 LoRA present:", os.path.isfile(os.path.join(lora, "adapter_config.json")))
print("  contents:", os.listdir(lora) if os.path.isdir(lora) else "(missing)")
print("PPO checkpoint present:", os.path.isdir(ckpt) and os.path.exists(os.path.join(ckpt, "progress.json")))

In [ ]:
# ── Stage 4b (faithful): PPO-tune the Stage-3 Llama (policy="llama") ──
# Needs model/llama_lora from Stage 3 on disk. Checkpoints to model/ppo_llama_ckpt (auto-resume
# on re-run). Heavier than pythia — tiny defaults "prove it runs"; raise steps/sample_size and use
# a bigger GPU for paper-grade. Same hand-rolled loop as Stage 4, just policy="llama".
import os, sys, importlib
prepare()
# deps already installed by Stage 3 (peft + bitsandbytes), whose model/llama_lora this consumes.
RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)
import ppo_core; importlib.reload(ppo_core)
asr = ppo_core.train_ppo(policy="llama", feature_type="Text",
                         steps=15, sample_size=256, batch_size=2)
os.chdir(REPO)
print("Stage 4b (llama) ASR:", asr)